In [131]:
import numpy as np
from skimage.measure import label, regionprops
from tifffile import imread, imwrite

In [133]:
def object_metrics(gt_files, pred_files, iou_threshold=0.5, min_pixels=100):

    metrics = {"TP": 0, "FP": 0, "FN": 0, "ious": [], "dices": []}
    frame_dices = []

    if len(gt_files) != len(pred_files):
        raise ValueError("Different numbers of ground truth and prediction files")

    for gt_file, pred_file in zip(gt_files, pred_files):

        gt_mask = imread(gt_file)
        pred_mask = imread(pred_file)

        if gt_mask.shape != pred_mask.shape:
            raise ValueError("Ground truth and prediction shapes do not match")

        # Spheroid = 0
        gt_bin = gt_mask == 0
        pred_bin = pred_mask == 0

        # Label individual spheroids
        gt = label(gt_bin)
        pred = label(pred_bin)

        # Remove small objects
        for i in range(1, gt.max() + 1):
            if np.sum(gt == i) < min_pixels:
                gt[gt == i] = 0

        for i in range(1, pred.max() + 1):
            if np.sum(pred == i) < min_pixels:
                pred[pred == i] = 0

        gt_props = regionprops(gt)
        pred_props = regionprops(pred)

        matched_gt = set()
        TP = 0
        FP = 0

        for pred_region in pred_props:

            pred_label = pred_region.label
            pred_mask_obj = pred == pred_label

            overlapping = gt[pred_mask_obj]
            overlapping = overlapping[overlapping != 0]

            if len(overlapping) == 0:
                FP += 1
                continue

            best_iou = 0
            best_gt = None

            for gt_label in np.unique(overlapping):

                if gt_label in matched_gt:
                    continue

                gt_mask_obj = gt == gt_label

                intersection = np.logical_and(
                    pred_mask_obj, gt_mask_obj
                ).sum()

                union = np.logical_or(
                    pred_mask_obj, gt_mask_obj
                ).sum()

                iou = intersection / union if union else 0.0

                if iou > best_iou:
                    best_iou = iou
                    best_gt = gt_label

            if best_iou >= iou_threshold:

                TP += 1
                matched_gt.add(best_gt)

                gt_match_obj = gt == best_gt

                intersection = np.logical_and(
                    pred_mask_obj, gt_match_obj
                ).sum()

                dice = (
                    2 * intersection /
                    (pred_mask_obj.sum() + gt_match_obj.sum())
                )

                metrics["ious"].append(best_iou)
                metrics["dices"].append(dice)

            else:
                FP += 1

        FN = len(gt_props) - len(matched_gt)

        metrics["TP"] += TP
        metrics["FP"] += FP
        metrics["FN"] += FN

        # Pixel-wise Dice across the entire frame
        intersection = np.logical_and(gt_bin, pred_bin).sum()

        frame_dice = (
            2 * intersection /
            (gt_bin.sum() + pred_bin.sum())
            if gt_bin.sum() + pred_bin.sum() else 0
        )

        frame_dices.append(frame_dice)

    precision = (
        metrics["TP"] / (metrics["TP"] + metrics["FP"])
        if metrics["TP"] + metrics["FP"] else 0
    )

    recall = (
        metrics["TP"] / (metrics["TP"] + metrics["FN"])
        if metrics["TP"] + metrics["FN"] else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    mean_iou = np.mean(metrics["ious"]) if metrics["ious"] else 0
    mean_object_dice = np.mean(metrics["dices"]) if metrics["dices"] else 0
    std_object_dice = np.std(metrics["dices"], ddof=1)
    frame_dice = np.mean(frame_dices) if frame_dices else 0
    frame_dice_std = np.std(frame_dices, ddof=1) if len(frame_dices) > 1 else 0

    return {
        "TP": metrics["TP"],
        "FP": metrics["FP"],
        "FN": metrics["FN"],
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_iou": mean_iou,
        "mean_object_dice": mean_object_dice,
        "object_dice_std": std_object_dice,
        "frame_dice": frame_dice,
        "frame_dice_std": frame_dice_std,
    }

In [ ]:
from pathlib import Path

#folders containing ground truth

folders = [] # list of folders containing ground truth and prediction files

gt_files = []
pred_files = []

for folder in folders:
    p = Path(folder)
    gt_files.extend(p.glob("*GT.tif"))
    pred_files.extend(p.glob("*RF.tif"))
print(len(gt_files), len(pred_files))

for i in folders:
    p = Path(i)
    print(p.exists())
    print(p.is_dir())

35 35
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [ ]:
object_metrics(gt_files, pred_files, iou_threshold=50, min_pixels=40)

{'TP': 133,
 'FP': 29,
 'FN': 111,
 'precision': 0.8209876543209876,
 'recall': 0.5450819672131147,
 'f1': 0.6551724137931034,
 'mean_iou': np.float64(0.5339161499295574),
 'mean_object_dice': np.float64(0.618355755887281),
 'object_dice_std': np.float64(0.34505460259244747),
 'frame_dice': np.float64(0.5203289252366953),
 'frame_dice_std': np.float64(0.2827184914821985)}